# 📊 Période 2 — Notebook 01 : Exploration des données MovieLens

**Objectif** : comprendre la structure et la distribution des données avant toute modélisation.

Fichiers analysés :
- `movies.dat`  → MovieID :: Title :: Genres
- `users.dat`   → UserID :: Gender :: Age :: Occupation :: Zip-code
- `ratings.dat` → UserID :: MovieID :: Rating :: Timestamp

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW = '../data/raw/'

movies  = pd.read_csv(RAW+'movies.dat',  sep='::', engine='python', encoding='latin-1',
                      names=['MovieID','Title','Genres'])
users   = pd.read_csv(RAW+'users.dat',   sep='::', engine='python', encoding='latin-1',
                      names=['UserID','Gender','Age','Occupation','Zip'])
ratings = pd.read_csv(RAW+'ratings.dat', sep='::', engine='python', encoding='latin-1',
                      names=['UserID','MovieID','Rating','Timestamp'])

print('='*50)
print(f'Films       : {len(movies):>8,}')
print(f'Utilisateurs: {len(users):>8,}')
print(f'Notes       : {len(ratings):>8,}')
print(f'Densité     : {len(ratings)/(len(movies)*len(users))*100:.2f}%')
print('='*50)

In [ ]:
# ── Distribution des notes ──────────────────────────────────────
print('Distribution des notes :')
dist = ratings['Rating'].value_counts().sort_index()
for rating, count in dist.items():
    bar = '█' * int(count / dist.max() * 30)
    print(f'  {rating}★  {bar}  {count:,} ({count/len(ratings)*100:.1f}%)')

print(f'\nMoyenne : {ratings.Rating.mean():.2f}')
print(f'Médiane : {ratings.Rating.median():.0f}')
print(f'Std     : {ratings.Rating.std():.2f}')

In [ ]:
# ── Analyse des films ────────────────────────────────────────────
print('Top 10 films les plus notés :')
top_movies = (ratings.groupby('MovieID')['Rating']
              .agg(['count','mean'])
              .merge(movies[['MovieID','Title']], on='MovieID')
              .sort_values('count', ascending=False)
              .head(10))
for _, r in top_movies.iterrows():
    print(f"  {r['count']:5,} notes  {r['mean']:.2f}★  {r.Title}")

print(f'\nFilms avec au moins 10 notes : {(ratings.groupby("MovieID").size() >= 10).sum()}')
print(f'Films jamais notés : {len(movies) - ratings["MovieID"].nunique()}')

In [ ]:
# ── Analyse des genres ───────────────────────────────────────────
genre_counts = (movies['Genres'].str.split('|')
                .explode()
                .value_counts())
print('Distribution des genres :')
for genre, count in genre_counts.items():
    bar = '█' * int(count / genre_counts.max() * 25)
    print(f'  {genre:20s} {bar}  {count}')

In [ ]:
# ── Analyse des utilisateurs ─────────────────────────────────────
print('Répartition par genre :')
print(users['Gender'].value_counts().to_string())

print('\nRépartition par tranche d\'âge :')
age_labels = {1:'<18', 18:'18-24', 25:'25-34', 35:'35-44', 45:'45-49', 50:'50-55', 56:'56+'}
for age, count in users['Age'].value_counts().sort_index().items():
    print(f'  {age_labels.get(age, age):8s} : {count}')

print(f'\nNotes par utilisateur :')
notes_per_user = ratings.groupby('UserID').size()
print(f'  Min     : {notes_per_user.min()}')
print(f'  Max     : {notes_per_user.max()}')
print(f'  Moyenne : {notes_per_user.mean():.1f}')
print(f'  Médiane : {notes_per_user.median():.0f}')

In [ ]:
# ── Biais par genre utilisateur ──────────────────────────────────
df = ratings.merge(users[['UserID','Gender']], on='UserID')
print('Note moyenne par genre utilisateur :')
print(df.groupby('Gender')['Rating'].agg(['mean','count']).round(3))

print('\nNote moyenne par tranche d\'âge :')
df2 = ratings.merge(users[['UserID','Age']], on='UserID')
age_means = df2.groupby('Age')['Rating'].mean().round(3)
for age, mean in age_means.items():
    print(f'  {age_labels.get(age, age):8s} : {mean:.3f}')

## 📝 Conclusions de l'exploration

1. **Les notes sont biaisées positivement** (moyenne ~3.7) → les utilisateurs notent surtout les films qu'ils aiment
2. **Déséquilibre de classes** : ~65% des notes sont ≥ 4 → Naïve Bayes devra gérer ce déséquilibre
3. **Les genres sont inégalement représentés** → certains genres auront moins de signal
4. **Densité faible** (~0.6%) → problème de cold start inévitable
5. **Features utiles identifiées** : Gender, Age, Occupation, Genres, statistiques film/utilisateur

→ Passer au notebook `02_feature_engineering.ipynb`